# Pattern #2: Tool Use - Bridge to the Outside World

**From-Scratch Implementation**

## Overview

Tools enable agents to interact with the external world. In our system:
- **Internal Memory**: RAG retrieval (agent cognition)
- **External Action**: web_search (agent tool)

This notebook demonstrates using `web_search` (Tavily) for real-world operational information.

In [ ]:
import sys
sys.path.append('..')

from utils import create_llm_provider, get_config
from tools import get_web_search_tool

# Initialize
config = get_config()
llm = create_llm_provider()
web_search = get_web_search_tool()

print(f"Using model: {config.get('model')}")
print(f"Web search provider: Tavily")
print(f"Timezone: {config.get('output.timezone')}")


### Validation (config from config/, web_search tool)

In [ ]:
assert config.get("model"), "config.get('model') should be set"
assert config.get("output.timezone"), "output.timezone should be set"
assert llm is not None and callable(getattr(web_search, "search", None)), "LLM and web_search ready"
print("✓ Setup valid: config, LLM, and web_search ready.")

## Baseline vs Tool-Enhanced Responses

Let's compare responses with and without web_search for operational queries.


In [ ]:
def baseline_response(query: str) -> dict:
    """Generate response without web search tool."""
    system_prompt = """
You are a healthcare assistant for Sri Lankan hospitals.
Provide helpful information about hospital services.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
    
    response = llm.generate(prompt=query, system_prompt=system_prompt)
    
    return {
        "response": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"],
        "sources": []
    }


def tool_enhanced_response(query: str) -> dict:
    """Generate response using web search tool."""
    import time
    
    start_time = time.time()
    
    # Step 1: Perform web search
    search_result = web_search.search(query)
    search_formatted = web_search.format_results(search_result)
    
    # Step 2: Generate response with search context
    system_prompt = """
You are a healthcare assistant with access to current web information.
Use the web search results to provide accurate, up-to-date operational information.
Always cite sources and include the checked timestamp.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
    
    prompt = f"""
User Question: {query}

Web Search Results:
{search_formatted}

Please provide a comprehensive answer using the search results. Include URLs for verification.
""".strip()
    
    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    
    end_time = time.time()
    total_latency = int((end_time - start_time) * 1000)
    
    return {
        "response": response["response"],
        "tokens": response["total_tokens"],
        "search_tokens": 0,  # Tavily doesn't charge tokens
        "latency_ms": total_latency,
        "search_latency_ms": search_result["latency_ms"],
        "sources": [r["url"] for r in search_result.get("results", [])]
    }


def compare_approaches(query: str):
    """Compare baseline vs tool-enhanced."""
    print("=" * 80)
    print("BASELINE (No Tools)")
    print("=" * 80)
    
    baseline = baseline_response(query)
    print(f"\n{baseline['response']}\n")
    print(f"[Tokens: {baseline['tokens']}, Latency: {baseline['latency_ms']}ms]")
    print(f"Sources: None\n")
    
    print("=" * 80)
    print("TOOL-ENHANCED (With web_search)")
    print("=" * 80)
    
    enhanced = tool_enhanced_response(query)
    print(f"\n{enhanced['response']}\n")
    print(f"[Tokens: {enhanced['tokens']}, Latency: {enhanced['latency_ms']}ms]")
    print(f"Search Latency: {enhanced['search_latency_ms']}ms")
    print(f"Sources ({len(enhanced['sources'])}):")
    for url in enhanced['sources'][:3]:
        print(f"  - {url}")
    
    print("\n" + "=" * 80)
    print("COMPARISON")
    print("=" * 80)
    print(f"Latency increase: {enhanced['latency_ms'] - baseline['latency_ms']}ms")
    print(f"Token increase: {enhanced['tokens'] - baseline['tokens']}")
    print(f"Source verification: {'Yes (URLs provided)' if enhanced['sources'] else 'No'}")
    
    return {"baseline": baseline, "enhanced": enhanced}


## Example 1: Hospital Hours Query


In [ ]:
query1 = "What are the OPD hours for Nawaloka Hospital Colombo?"
result1 = compare_approaches(query1)


## Example 2: Emergency Contact


In [ ]:
query2 = "What is the emergency hotline for Asiri Hospital?"
result2 = compare_approaches(query2)


## Pattern Summary

Tools extend agent capabilities to interact with the external world.

**Key Insights:**
- **Baseline**: LLM generates from training data (may be outdated/generic)
- **Tool-Enhanced**: Real-time, verified information with sources
- **web_search is the ONLY tool**: RAG is internal memory, not a tool

**When to use web_search:**
- Operational information (hours, contacts, addresses)
- Current announcements or changes
- Verification of time-sensitive facts
- Location-specific details

**Architecture:**
```
Internal Cognition: RAG retrieval (memory)
External Action: web_search (tool)
```

This separation is crucial for understanding agentic systems.
